<a href="https://colab.research.google.com/github/anangshachatterjee/anangsha-chatterjee-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one report-date observation for one pseudonymized client and one pseudonymized content item.

**Primary table:** `fact_content_daily_performance`.

**Development time window:** March 2026 (`month=2026-03`).

I will use March 2026 as the mid-panel development month. I will not use the final-month `_sample` for developing label logic because it represents the latest full month and will be treated as a sealed test period.

**Prediction/ranking goal:** I want to rank content pages by their likelihood of showing declining search performance, using only information available before the decision moment.

**Deliberate exclusion:** I will exclude fields derived from the future outcome or fields that would only be available after the decision, because they could introduce target leakage.

In [ ]:
from huggingface_hub import login
login()

from datasets import load_dataset
performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

performance_df = performance["train"].to_pandas()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 8.93MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Features

I will use historical search-performance variables that were available before the decision moment. I will keep the feature set to a maximum of five variables.

### Label

The label/proxy will represent a future decline in search performance. It will be defined from an outcome window after the feature/decision window.

### Context

Pseudonymized client and content identifiers will be used for grouping, joining, and verification rather than treated as meaningful predictive signals.

### Excluded

I will exclude fields that directly encode the outcome, are calculated from the future window, or would only be known after the decision. These fields could cause target leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
grain_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (
            report_date,
            client_id,
            content_id
        )) AS distinct_grain
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    """
).df()

grain_check

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



### Limitation: Unequal historical coverage

The warehouse is an unbalanced panel, so different clients have different depths of historical data. This means that observations from different clients may not represent equally complete histories.

The data is observational rather than experimental. It can show measured patterns and support ranking decisions, but it cannot by itself prove that a particular content change caused a future improvement in search performance.

The fixed 90-day query windows can also overlap reporting periods, so feature and outcome windows must be defined carefully to avoid using future information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.